# Classifier for PathMNIST

This notebook trains a multiclass classifier on PathMNIST in PyTorch. We begin with a simple MLP, before moving on to a CNN. 

## Multilayer Perceptron

In [32]:
import numpy as np
import torch
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torchmetrics

load the data file w/ numpy, then turn all the of features into a seperate tensor for train/test/val respectively

In [2]:
data = np.load('data/pathmnist.npz') # load file using numpy

In [3]:
tensors = {name: torch.from_numpy(data[name]) for name in data.files}

In [4]:
for name, t in tensors.items():
    print(f"{name:15} {tuple(t.shape)} {t.dtype}")

train_images    (89996, 28, 28, 3) torch.uint8
val_images      (10004, 28, 28, 3) torch.uint8
test_images     (7180, 28, 28, 3) torch.uint8
train_labels    (89996, 1) torch.uint8
val_labels      (10004, 1) torch.uint8
test_labels     (7180, 1) torch.uint8


So now we have a dict of train/test/val images (inputs) and labels (target) tensors. We can now properly turn them into our data.

We also need to change the pixel values in each tensor to a tensor float32 and scale them by dividing by 255 to compress everything in a 0-1 range


In [5]:
for name in tensors.keys():
    tensors[name] = tensors[name]

In [6]:
X_train = tensors['train_images'].to(torch.float32) / 255
X_val = tensors['val_images'].to(torch.float32) / 255
X_test = tensors['test_images'].to(torch.float32) / 255

y_train = torch.flatten(tensors['train_labels'])
y_val = torch.flatten(tensors['val_labels'])
y_test = torch.flatten(tensors['test_labels'])

Now we have our data setup. 

Each image is a 28x28 grid of pixels, with each pixel value represented as a 3 entry vector reprenting it's RGB colour channel.

We also flattened the targets, so it's 1 vector with N entires for our loss function to work

For our future CNN, we should rework this so it's 3 28x28 grids representing each colour instead, but not needed rn since an MLP flattens this image

In [7]:
X_train = torch.permute(X_train, (0, 3, 1, 2))
X_val = torch.permute(X_val, (0, 3, 1, 2))
X_test = torch.permute(X_test, (0, 3, 1, 2))

In [8]:
X_train.size()

torch.Size([89996, 3, 28, 28])

Now we need to create our DataSet objects and our data loaders

In [9]:

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)

Now that we have our data loaders, we can build the classification MLP

We will create a custom module, but it's really just a sequential module, custom is just cleaner and easier to fine tune hyperparams

n_inputs is the # of features (3x28x28), n_classes is the # of output classes (9)

note that we don't apply softmax (converts logits to probs) on the output layer, so the model outputs logits (raw class scores)

this is because we use cross entropy loss as our loss function, which computes loss straight from the logits

In [10]:
class MLPClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(), # flatten the 3x28x28 image for matmul w/ weights to work
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )

    def forward(self, X):
        return self.mlp(X)

We have our model now, so now we can focus on training it. 

We can pick our loss function, (Cross Entropy), our learning rate, and our optimizer (SGD, but we can use it with batch gd)
- optimizer adjusts the models parameters according to the learning rate and the loss 

In [29]:
device = 'mps'
torch.manual_seed(42)

model = MLPClassifier(n_inputs=3*28*28, n_hidden1=500, n_hidden2=200, n_classes=9)
model = model.to(device)

xentropy = nn.CrossEntropyLoss()

learning_rate = 1e-3

# change optimizer from SGD to Adam 
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

Now that we have our model, loss function, and optimizer we need to write our training loop

In [30]:
def train_one_epoch(model, optimizer, criterion, train_loader, epoch=1):
    model.train()
    total_loss = 0.
    for X_batch, y_batch, in train_loader: 
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print(f'Loss for epoch {epoch}: {total_loss / len(train_loader)}')

def train(train_one_epoch, model, optimizer, criterion, train_loader, n_epochs):
    for epoch in range(n_epochs):
        train_one_epoch(model, optimizer, criterion, train_loader, epoch + 1)



Now let's train our model on the gpu

In [31]:
n_epochs = 20
train(train_one_epoch, model, optimizer, xentropy, train_loader, n_epochs)

Loss for epoch 1: 1.7031039986759424
Loss for epoch 2: 1.4339012510397218
Loss for epoch 3: 1.3614385910332203
Loss for epoch 4: 1.3254010264169087
Loss for epoch 5: 1.297241936861114
Loss for epoch 6: 1.2714771723205394
Loss for epoch 7: 1.2582035069777207
Loss for epoch 8: 1.2333287760954013
Loss for epoch 9: 1.2150243518196724
Loss for epoch 10: 1.2021127328784629
Loss for epoch 11: 1.1828338830647143
Loss for epoch 12: 1.161768732423132
Loss for epoch 13: 1.1435978427867999
Loss for epoch 14: 1.1224766522222622
Loss for epoch 15: 1.1107684179971165
Loss for epoch 16: 1.0996010289950804
Loss for epoch 17: 1.075512359172783
Loss for epoch 18: 1.0586199663918128
Loss for epoch 19: 1.0490040386264974
Loss for epoch 20: 1.0301594356582924


We tried SGD, but Adam seems to be the better optimizer for us

Now let's write a function to evaluate the model

In [34]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)

    return aggregate_fn(torch.stack(metrics))

Choose the Accuracy streaming metric, move to gpu. mean works fine as our aggregate for now 
- computes by comparing the idx of the highest value logit, and comparing it to the corresponding label.

In [35]:
accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=9).to(device)

In [36]:
evaluate(model, val_loader, accuracy)

tensor(0.5628, device='mps:0')

Accuracy of the model is not terrible for a simple MLP. Let's try fine tuning it before moving on to something more complex. 

In [38]:
import optuna

def objective(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    n_hidden1 = trial.suggest_int('n_hidden1', 20, 300)
    n_hidden2 = trial.suggest_int('n_hidden2', 20, 300)
    model = MLPClassifier(n_inputs=3*28*28, n_hidden1=n_hidden1, n_hidden2=n_hidden2, n_classes=9).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    train(train_one_epoch, model, optimizer, xentropy, train_loader, n_epochs)
    validation_accuracy = evaluate(model, val_loader, accuracy)
    return validation_accuracy

/Users/zainbabar/code/shiftmon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [39]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=10)

[I 2026-09-26 22:46:15,915] A new study created in memory with name: no-name-500adcd9-eac5-4ad3-a09a-0cb1cef36176


Loss for epoch 1: 1.7695269061421806
Loss for epoch 2: 1.5510011361065237
Loss for epoch 3: 1.415072388439016
Loss for epoch 4: 1.3610436366024343
Loss for epoch 5: 1.3309631533920765
Loss for epoch 6: 1.3150820088657467
Loss for epoch 7: 1.29723687317561
Loss for epoch 8: 1.2847950714216991
Loss for epoch 9: 1.2696165467866443
Loss for epoch 10: 1.259237079796466
Loss for epoch 11: 1.2477699646895581
Loss for epoch 12: 1.2344693139364773
Loss for epoch 13: 1.2184178089214996
Loss for epoch 14: 1.1970550140196627
Loss for epoch 15: 1.1777279540388421
Loss for epoch 16: 1.161302051083608
Loss for epoch 17: 1.1417008317498998
Loss for epoch 18: 1.1343217255039648
Loss for epoch 19: 1.1134373668411917
Loss for epoch 20: 1.1028854561630974


[I 2026-09-26 22:46:53,703] Trial 0 finished with value: 0.5575158596038818 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden1': 287, 'n_hidden2': 225}. Best is trial 0 with value: 0.5575158596038818.


Loss for epoch 1: 1.6853041462600231
Loss for epoch 2: 1.3809182416986336
Loss for epoch 3: 1.31046128120612
Loss for epoch 4: 1.2797225414859978
Loss for epoch 5: 1.240989153442735
Loss for epoch 6: 1.2241606145081194
Loss for epoch 7: 1.1928101286122745
Loss for epoch 8: 1.1783757700838826
Loss for epoch 9: 1.1500963299450548
Loss for epoch 10: 1.134942902844738
Loss for epoch 11: 1.1202732714909045
Loss for epoch 12: 1.0983222695067525
Loss for epoch 13: 1.0884826429696246
Loss for epoch 14: 1.075135546228425
Loss for epoch 15: 1.0653423164886506
Loss for epoch 16: 1.0467375591397285
Loss for epoch 17: 1.045460307123986
Loss for epoch 18: 1.0277087839997627
Loss for epoch 19: 1.0389191699637608


[I 2026-09-26 22:47:27,270] Trial 1 finished with value: 0.6190071105957031 and parameters: {'learning_rate': 0.0024810409748678114, 'n_hidden1': 63, 'n_hidden2': 63}. Best is trial 1 with value: 0.6190071105957031.


...

Loss for epoch 20: 1.3930846191942692
Loss for epoch 1: 1.7537110793319615
Loss for epoch 2: 1.5304830365560271
Loss for epoch 3: 1.3999556993896312
Loss for epoch 4: 1.3462598176503724
Loss for epoch 5: 1.322074772620743
Loss for epoch 6: 1.3082107090475885
Loss for epoch 7: 1.2865125045857646
Loss for epoch 8: 1.2749836654825644
Loss for epoch 9: 1.2689657772637226
Loss for epoch 10: 1.250356823713942
Loss for epoch 11: 1.2425130072303794
Loss for epoch 12: 1.233944696543569
Loss for epoch 13: 1.2212874137542464
Loss for epoch 14: 1.2060154088010842
Loss for epoch 15: 1.1817143054848367
Loss for epoch 16: 1.162951796992936
Loss for epoch 17: 1.1557231674986808
Loss for epoch 18: 1.13512118880383
Loss for epoch 19: 1.1213914141566916


[I 2026-09-26 22:51:45,919] Trial 8 finished with value: 0.5371835231781006 and parameters: {'learning_rate': 0.0006672367170464204, 'n_hidden1': 240, 'n_hidden2': 76}. Best is trial 1 with value: 0.6190071105957031.


Loss for epoch 20: 1.1104931443591008
Loss for epoch 1: 1.7346004171466285
Loss for epoch 2: 1.5707254914397544
Loss for epoch 3: 1.4171661052175544
Loss for epoch 4: 1.358800869181075
Loss for epoch 5: 1.3324527996168896
Loss for epoch 6: 1.3153336266563698
Loss for epoch 7: 1.3052403281696818
Loss for epoch 8: 1.2938392719423228
Loss for epoch 9: 1.2800325040112843
Loss for epoch 10: 1.263784821822562
Loss for epoch 11: 1.2558902069418267
Loss for epoch 12: 1.2392365909604863
Loss for epoch 13: 1.2337240045043556
Loss for epoch 14: 1.22492724199864
Loss for epoch 15: 1.210379023104906
Loss for epoch 16: 1.2006420522758907
Loss for epoch 17: 1.1946424427018925
Loss for epoch 18: 1.1887837277556008
Loss for epoch 19: 1.1748238592831926


[I 2026-09-26 22:52:23,476] Trial 9 finished with value: 0.5161194801330566 and parameters: {'learning_rate': 0.0011400863701127321, 'n_hidden1': 186, 'n_hidden2': 33}. Best is trial 1 with value: 0.6190071105957031.


Loss for epoch 20: 1.153978895988654


In [40]:
study.best_params

{'learning_rate': 0.0024810409748678114, 'n_hidden1': 63, 'n_hidden2': 63}

In [41]:
study.best_value

0.6190071105957031

This looks like this is the best it's gonna get with a basic MLP, now let's try something better

## Convolutional Neural Network